In [1]:
!pip install python-dotenv groq


  Obtaining dependency information for groq from https://files.pythonhosted.org/packages/3f/f6/c3d8ea3194d7d7d8292edd4dbb4b6832d29c2eab04887dda4a742e83902c/groq-1.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for anyio<5,>=3.5.0 from https://files.pythonhosted.org/packages/12/b8/4bd346e22b28902df4d651910f5242c28d84e4a5c2435ca5c3f797ed7e2e/anyio-4.15.1-py3-none-any.whl.metadata
  Obtaining dependency information for distro<2,>=1.7.0 from https://files.pythonhosted.org/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
  Obtaining dependency information for pydantic<3,>=1.9.0 from https://files.pythonhosted.org/packages/2d/eb/9146591cc819d040475bf7f2be786710c7f7eb8083693bf859728da2ca9c/pydantic-2.14.0-p


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os 

In [3]:

from dotenv import load_dotenv

# This will look for your .env file and load the keys
load_dotenv()

# Verify if it finds your key
print("API Key Loaded Successfully:", os.environ.get("GROQ_API_KEY") is not None)


API Key Loaded Successfully: True


In [4]:
from groq import Groq

# Initialize the Groq client (it automatically picks up your GROQ_API_KEY from the environment)
client = Groq()

def ask_llm(prompt_text):
    """Sends a prompt to Groq and returns the generated text response."""
    response = client.chat.completions.create(
        # We will use llama3-8b-8192 as it is incredibly fast and great for testing prompts
        model="openai/gpt-oss-20b", 
        messages=[
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0.7, # Controls creativity (lower is more focused, higher is more creative)
    )
    # Extract and return the text message from the response object
    return response.choices[0].message.content


In [5]:
# The basic prompt with no extra instructions
prompt1 = "tell me about python?"

print("--- Running Test 1 (No Prompt Engineering) ---")
# Call our function and print the response
response1 = ask_llm(prompt1)
print(response1)


--- Running Test 1 (No Prompt Engineering) ---
## Python in a Nutshell

Python is a general‑purpose, high‑level programming language that has become one of the most popular languages in the world. It was created by **Guido van Rossum** and first released in **1991**. The language emphasizes readability, simplicity, and developer productivity, which has made it a favorite for beginners and professionals alike.

---

### 1. Core Philosophy

| Principle | What it means |
|-----------|---------------|
| **Explicit is better than implicit** | Clear, self‑documenting code |
| **Simple is better than complex** | Prefer straightforward solutions |
| **Readability counts** | Code should be easy to read and understand |
| **There should be one— and preferably only one —obvious way to do it** | Reduce ambiguity |

These ideas are captured in the “Zen of Python” (`import this`).

---

### 2. Key Features

| Feature | Why it matters |
|---------|----------------|
| **Interpreted, dynamic typing** |

In [6]:
# The engineered prompt assigning a persona and explicit structural constraints
prompt2 = """you are expert in python coding language.help me understand what is python and explain in 5 points."""

print("--- Running Test 2 (With Prompt Engineering) ---")
# Call our function with the engineered prompt
response2 = ask_llm(prompt2)
print(response2)


--- Running Test 2 (With Prompt Engineering) ---
**Python in a nutshell – 5 key points**

| # | What it is | Why it matters |
|---|------------|----------------|
| 1 | **High‑level, interpreted language** | You write code once and run it on any platform without compiling. The interpreter translates Python into machine code at runtime, making rapid prototyping a breeze. |
| 2 | **Readable, English‑like syntax** | Python uses indentation instead of braces, and its keywords (`def`, `for`, `if`, `import`) read like plain English. This reduces boilerplate and makes the code self‑documenting, which is great for beginners and large teams. |
| 3 | **Extensive standard library + ecosystem** | From `datetime` and `json` to `asyncio` and `multiprocessing`, the stdlib covers most everyday tasks. The Python Package Index (PyPI) hosts over 300 000 third‑party packages, so almost any domain (web, data science, AI, automation) has mature libraries. |
| 4 | **Multi‑paradigm support** | Python lets you 

In [7]:
def ask_llm_with_temp(prompt_text, custom_temp):
    """Sends a prompt to Groq with a specific temperature setting."""
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", 
        messages=[{"role": "user", "content": prompt_text}],
        temperature=custom_temp,
    )
    # Added [0] here to properly pick the first response option
    return response.choices[0].message.content

# Run the test query again
creative_prompt = "Write a one-sentence slogan for a futuristic coffee shop."

print("--- Testing Low Temperature (0.0) - Factual & Direct ---")
print(ask_llm_with_temp(creative_prompt, custom_temp=0.0))

print("\n--- Testing High Temperature (1.2) - Creative & Random ---")
print(ask_llm_with_temp(creative_prompt, custom_temp=1.2))


--- Testing Low Temperature (0.0) - Factual & Direct ---
Sip the future, one cup at a time.

--- Testing High Temperature (1.2) - Creative & Random ---
“Brewing tomorrow, one cup at a time.”


In [8]:
def ask_llm_with_tokens(prompt_text, tokens_limit):
    """Sends a prompt to Groq with a strict max token limit."""
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", 
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0.7,
        max_tokens=tokens_limit  # <--- Setting the hard token generation limit here
    )
    return response.choices[0].message.content

# Test prompt asking for a long explanation
long_prompt = "Explain how gravity works in detail."

print("--- Testing Strict Max Tokens (15 Tokens) ---")
print(ask_llm_with_tokens(long_prompt, tokens_limit=15))

print("\n--- Testing Generous Max Tokens (100 Tokens) ---")
print(ask_llm_with_tokens(long_prompt, tokens_limit=100))


--- Testing Strict Max Tokens (15 Tokens) ---


--- Testing Generous Max Tokens (100 Tokens) ---



In [9]:
# Step 3: Placeholder Prompts (Templates)
# A placeholder prompt uses Python's .format() to easily swap out inputs into a template

prompt_template = """You are a helpful teaching assistant. 
Explain the concept of {TOPIC} to a student.
Constraint: Your explanation must be extremely short, using exactly {WORDS} words or less."""

print("--- Testing Template with strict 10-word constraint ---")
# Injecting "Gravity" and "10" into the placeholder template
formatted_prompt1 = prompt_template.format(TOPIC="Gravity", WORDS="10")
response1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": formatted_prompt1}],
    temperature=0.7
)
print(response1.choices[0].message.content)


print("\n--- Testing Template with a longer 50-word constraint ---")
# Injecting "Gravity" and "50" into the same template
formatted_prompt2 = prompt_template.format(TOPIC="Gravity", WORDS="50")
response2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": formatted_prompt2}],
    temperature=0.7
)
print(response2.choices[0].message.content)


--- Testing Template with strict 10-word constraint ---
Gravity pulls objects together, keeping planets, people, and stars bound.

--- Testing Template with a longer 50-word constraint ---
Gravity is a force that pulls objects toward each other. Earth’s mass attracts everything nearby, keeping us on the ground. The heavier an object, the stronger its pull. This attraction also keeps planets orbiting stars and shapes the motion of everything in space, and explains why we fall.


In [10]:
# 1. Create a reusable prompt template with two placeholders: {ROLE} and {TOPIC}
template = "You are a world-class expert in {ROLE}. Explain {TOPIC} in exactly one short sentence."

print("--- Running Test A ---")
# 2. Fill the placeholders using .format()
prompt_a = template.format(ROLE="Cooking", TOPIC="how to chop an onion safely")
print("Generated Prompt A:", prompt_a)

# 3. Send it to the LLM
response_a = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_a}],
    temperature=0.5
)
print("AI Response A:", response_a.choices[0].message.content)


print("\n--- Running Test B (Using the SAME template!) ---")
# 4. Reuse the exact same template, but with different inputs!
prompt_b = template.format(ROLE="Space Science", TOPIC="why Mars is red")
print("Generated Prompt B:", prompt_b)

response_b = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_b}],
    temperature=0.5
)
print("AI Response B:", response_b.choices[0].message.content)


--- Running Test A ---
Generated Prompt A: You are a world-class expert in Cooking. Explain how to chop an onion safely in exactly one short sentence.
AI Response A: Keep the root end intact and slice with a sharp knife, tucking your fingers in for a secure grip.

--- Running Test B (Using the SAME template!) ---
Generated Prompt B: You are a world-class expert in Space Science. Explain why Mars is red in exactly one short sentence.
AI Response B: Mars appears red because iron oxide (rust) dust coats its surface.


In [11]:
zero_shot_prompt = """
Classify the sentiment of the following review as either Positive or Negative.

Review: "The delivery took two weeks and the box arrived damaged."
Sentiment:
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0.0
)
print("--- Zero-Shot Output ---")
print(response.choices[0].message.content)


--- Zero-Shot Output ---
Sentiment: Negative


In [13]:
one_shot_prompt = """
Classify the sentiment of the following review as either Positive or Negative.

Review: "I absolutely love this vacuum! Best purchase ever."
Sentiment: Positive

Review: "The delivery took two weeks and the box arrived damaged."
Sentiment:
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": one_shot_prompt}],
    temperature=0.0
)
print("--- One-Shot Output ---")
print(response.choices[0].message.content)


--- One-Shot Output ---
Sentiment: Negative


In [14]:
few_shot_prompt = """
Classify the sentiment of the following review as either Positive, Neutral, or Negative.

Review: "I absolutely love this vacuum! Best purchase ever."
Sentiment: Positive

Review: "It works fine, nothing special but gets the job done."
Sentiment: Neutral

Review: "Stopped working after two days. Do not buy."
Sentiment: Negative

Review: "The delivery took two weeks and the box arrived damaged."
Sentiment:
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0.0
)
print("--- Few-Shot Output ---")
print(response.choices[0].message.content)


--- Few-Shot Output ---
Sentiment: Negative
